# Q-Learning Gridworld Dynamic Environment Lab


## Purpose

This lab introduces Q-learning in a small dynamic grid environment.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

grid_size = 5
n_states = grid_size * grid_size
n_actions = 4

actions = {0: "up", 1: "down", 2: "left", 3: "right"}

goal_state = 24
hazard_state = 12

def state_to_position(state):
    return divmod(state, grid_size)

def position_to_state(row, col):
    return row * grid_size + col

def step_environment(state, action, episode):
    row, col = state_to_position(state)

    if action == 0:
        row = max(0, row - 1)
    elif action == 1:
        row = min(grid_size - 1, row + 1)
    elif action == 2:
        col = max(0, col - 1)
    elif action == 3:
        col = min(grid_size - 1, col + 1)

    next_state = position_to_state(row, col)
    goal_reward = 10.0 if episode < 400 else 6.0

    if next_state == goal_state:
        return next_state, goal_reward, True
    if next_state == hazard_state:
        return next_state, -8.0, False
    return next_state, -0.1, False

In [ ]:
alpha = 0.15
gamma = 0.95
epsilon = 0.20
episodes = 800
max_steps = 80

Q = np.zeros((n_states, n_actions))
rows = []

for episode in range(episodes):
    state = 0
    total_reward = 0.0
    done = False

    for step_index in range(max_steps):
        if rng.random() < epsilon:
            action = int(rng.integers(n_actions))
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, done = step_environment(state, action, episode)

        td_target = reward + gamma * np.max(Q[next_state])
        td_error = td_target - Q[state, action]
        Q[state, action] += alpha * td_error

        total_reward += reward
        state = next_state

        if done:
            break

    rows.append({
        "episode": episode,
        "total_reward": total_reward,
        "reached_goal": done,
        "phase": "early_reward" if episode < 400 else "shifted_reward",
    })

results = pd.DataFrame(rows)
results.groupby("phase").agg(
    mean_reward=("total_reward", "mean"),
    goal_rate=("reached_goal", "mean"),
)

## Interpretation

The reward shift tests whether the learned policy remains effective when the environment changes.